# Módulo 5: Cálculo II
## Lección 7: Integrales de Línea en Campos Escalares y Vectoriales

Las **integrales de línea** son herramientas de acumulación matemática a lo largo de una trayectoria curvilínea en el espacio. Encuentran aplicaciones directas e indispensables en la física teórica y la ingeniería clásica, permitiendo formular el concepto de trabajo mecánico realizado por una fuerza a lo largo de un camino, calcular la circulación de un fluido viscoso o modelar fenómenos electromagnéticos gobernados por las leyes de Faraday y Ampère.

En esta lección, estudiaremos formalmente las integrales de línea para campos escalares y vectoriales. Analizaremos las implicaciones físicas de los campos conservativos y la independencia de camino, deduciendo la existencia de funciones potenciales y el teorema fundamental de las integrales de línea. Adicionalmente, demostraremos la dependencia del camino para fuerzas no conservativas (como la fricción) e identificaremos erratas matemáticas comunes en los textos de estudio mediante simulaciones interactivas y cálculo simbólico con Python.

### Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Definir y calcular la integral de línea** de un campo escalar a lo largo de curvas suaves en $\mathbb{R}^2$ y $\mathbb{R}^3$.
2. **Calcular el trabajo mecánico y la circulación** de un fluido mediante integrales de línea de campos vectoriales.
3. **Determinar si un campo es conservativo** evaluando su rotacional y extraer su función potencial exacta.
4. **Aplicar el Teorema Fundamental de las integrales de línea** en campos conservativos para simplificar los cálculos.
5. **Contrastar el comportamiento físico de sistemas conservativos** (como la gravedad) frente a **no conservativos** (como la fricción) a través de simulaciones de dependencia del camino.
6. **Implementar integradores de línea didácticos (Riemann)** y validar sus resultados mediante el motor de cálculo simbólico de `SymPy`.

### 1. Integrales de Línea de Campos Escalares

Sea $f: \mathbb{R}^n \to \mathbb{R}$ una función escalar continua en una región del espacio que contiene a una curva suave $C$ parametrizada por $\mathbf{r}(t) = \langle x(t), y(t), z(t) \rangle$ para $t \in [a, b]$. La **integral de línea de $f$ a lo largo de $C$** se define como:

$$\int_C f ds = \int_a^b f(\mathbf{r}(t)) \|\mathbf{r}'(t)\| dt$$

donde $ds = \|\mathbf{r}'(t)\| dt$ representa el diferencial de longitud de arco.

#### Corrección de Errata de Texto Guía (Integración de Campo Escalar)
> [!IMPORTANT]
> **ERRATA DETECTADA EN EL TEXTO GUÍA:**
> En el material de estudio (Ejemplo 1, pág. 4), se solicita calcular la integral del campo $f(x, y) = x^2 + y^2$ a lo largo de la curva $\mathbf{r}(t) = \langle t, t^2 \rangle$ para $t \in [0, 1]$. El texto plantea la integral:
> 
> $$\int_C f ds = \int_0^1 (t^2 + t^4) \sqrt{1 + 4t^2} dt$$
> 
> Para resolverla, el texto aplica erróneamente una sustitución de la forma $u = 1+4t^2 \implies du = 8t dt$, afirmando que la integral se simplifica de forma directa a:
> 
> $$\frac{1}{8} \int_1^5 (u^{3/2} - u^{1/2}) du \quad \text{[SUSTITUCIÓN INCORRECTA DEL TEXTO]}$$
> 
> esto es un **error algebraico grave** porque el integrando $(t^2 + t^4) \sqrt{1 + 4t^2} dt$ no posee el factor $t$ requerido para completar el diferencial de la sustitución ($du = 8t dt$).
> 
> Evaluando numéricamente ambas expresiones:
> - La integral que resolvió el texto da: **$\approx 1.8967$**
> - La integral real verdadera del campo da: **$\approx 0.99639$**
> 
> Para resolver analíticamente de forma correcta la integral original se requiere una sustitución trigonométrica ($2t = \tan\theta$) que conduce a integrales de secantes y tangentes de potencias impares. Evaluaremos este comportamiento numéricamente y lo validaremos con SymPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from mpl_toolkits.mplot3d import Axes3D

# Configuración de estilo visual
plt.style.use('seaborn-v0_8-whitegrid')

def integral_linea_escalar_discreta(f, r, t0, tf, N):
    t = np.linspace(t0, tf, N + 1)
    puntos = np.array([r(ti) for ti in t])
    
    suma = 0.0
    for i in range(N):
        t_mid = 0.5 * (t[i] + t[i+1])
        p_mid = r(t_mid)
        val_f = f(*p_mid)
        
        # ds = ||r(t_{i+1}) - r(t_i)||
        delta_r = puntos[i+1] - puntos[i]
        ds = np.linalg.norm(delta_r)
        
        suma += val_f * ds
    return suma

# f(x, y) = x^2 + y^2 y curva r(t) = (t, t^2)
def f_campo(x, y):
    return x**2 + y**2

def r_curva(t):
    return np.array([t, t**2])

# El valor real correcto de la integral es ~0.9963919
v_real = 0.9963919453
v_texto_erroneo = 1.8967232  # valor resuelto incorrectamente por el texto guía

print("VERIFICACIÓN DE INTEGRAL ESCALAR:")
for N in [10, 50, 200, 1000]:
    aprox = integral_linea_escalar_discreta(f_campo, r_curva, 0.0, 1.0, N)
    print(f"  N = {N:4d} | Aprox: {aprox:.6f} | Error vs Real: {abs(aprox - v_real):.2e} | Error vs Texto: {abs(aprox - v_texto_erroneo):.2e}")

### 2. Integrales de Línea de Campos Vectoriales

Sea $\mathbf{F}: \mathbb{R}^n \to \mathbb{R}^n$ un campo vectorial continuo a lo largo de una curva suave $C$ parametrizada por $\mathbf{r}(t)$ para $t \in [a, b]$. La **integral de línea de $\mathbf{F}$ a lo largo de $C$** es:

$$\int_C \mathbf{F} \cdot d\mathbf{r} = \int_a^b \mathbf{F}(\mathbf{r}(t)) \cdot \mathbf{r}'(t) dt$$

Esta integral calcula la acumulación de la componente del campo paralela a la trayectoria. En física:
- Si el campo es una fuerza, representa el **trabajo mecánico** realizado por la fuerza sobre la partícula.
- Si el campo es de velocidades de un fluido, representa la **circulación** del fluido a lo largo del camino cerrado.

Graficaremos el campo vectorial $\mathbf{F}(x, y, z) = \langle y, -x, z \rangle$ y la hélice espacial $\mathbf{r}(t) = \langle \cos t, \sin t, t \rangle$.

In [ ]:
t_vals = np.linspace(0, 2*np.pi, 100)
x_c = np.cos(t_vals)
y_c = np.sin(t_vals)
z_c = t_vals

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

# Graficar la curva hélice
ax.plot(x_c, y_c, z_c, 'b-', linewidth=3, label='Hélice Circular $\\mathbf{r}(t)$')

# Trazar algunos vectores del campo F = (y, -x, z) sobre la curva
indices_muestreo = np.linspace(0, 99, 15, dtype=int)
for idx in indices_muestreo:
    px, py, pz = x_c[idx], y_c[idx], z_c[idx]
    # Evaluar campo
    vx, vy, vz = py, -px, pz
    # Normalizar longitud para visualización limpia
    mag = np.hypot(vx, np.hypot(vy, vz))
    ax.quiver(px, py, pz, vx/mag*0.4, vy/mag*0.4, vz/mag*0.4, color='red', alpha=0.8)

ax.set_title('Trayectoria Hélice en el Campo Vectorial $\\mathbf{F}(x,y,z) = \\langle y, -x, z \\rangle$')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### 3. Independencia del Camino y Campos Conservativos

Un campo vectorial $\mathbf{F}$ se dice que es **conservativo** si es el gradiente de alguna función potencial escalar $\phi$:

$$\mathbf{F} = \nabla \phi$$

**Teorema Fundamental de las Integrales de Línea:** Si $\mathbf{F} = \nabla \phi$ es conservativo, entonces la integral de línea a lo largo de cualquier curva suave $C$ que conecte los puntos $A$ y $B$ depende únicamente de los extremos, siendo independiente del camino:

$$\int_C \mathbf{F} \cdot d\mathbf{r} = \phi(B) - \phi(A)$$

En tres dimensiones, un campo diferenciable en una región simplemente conexa es conservativo si y solo si su rotacional es nulo:

$$\nabla \times \mathbf{F} = \mathbf{0}$$

A continuación, utilizaremos `SymPy` para verificar de forma simbólica la naturaleza conservativa del campo:

$$\mathbf{F}(x, y, z) = \langle y^2, \ 2xy, \ z \rangle$$

y evaluar su integral de línea desde $A(1, 1, 1)$ hasta $B(2, 3, 4)$ por su potencial.

In [ ]:
x_s, y_s, z_s = sp.symbols('x y z', real=True)
F_vec = sp.Matrix([y_s**2, 2*x_s*y_s, z_s])

# Rotacional
rot_F = sp.Matrix([
    F_vec[2].diff(y_s) - F_vec[1].diff(z_s),
    F_vec[0].diff(z_s) - F_vec[2].diff(x_s),
    F_vec[1].diff(x_s) - F_vec[0].diff(y_s)
])

print("Rotacional de F (debe ser [0, 0, 0]^T):")
sp.pprint(rot_F.T)

# Encontrar la función potencial phi
phi = sp.integrate(F_vec[0], x_s) + sp.integrate(F_vec[2], z_s)
print("\nFunción potencial escalar phi(x, y, z):")
sp.pprint(phi)

# Evaluar la integral usando la función potencial
phi_A = phi.subs({x_s: 1, y_s: 1, z_s: 1})
phi_B = phi.subs({x_s: 2, y_s: 3, z_s: 4})
integral = phi_B - phi_A
print(f"\nTrabajo total: phi(2, 3, 4) - phi(1, 1, 1) = {phi_B} - {phi_A} = {integral} (24.5)")

### 4. Interpretación Física: Sistemas Conservativos vs. No Conservativos

- **Campos Conservativos (Gravedad y Electrostática):** El trabajo mecánico necesario para mover una masa o carga de un punto $A$ a un punto $B$ es independiente del trayecto recorrido. No hay pérdidas por disipación y la energía total del sistema mecánico se conserva.
- **Campos No Conservativos (Fricción y Fluidos):** El trabajo realizado por una fuerza de rozamiento viscoso o fricción de deslizamiento depende directamente del camino recorrido (a mayor longitud del camino, mayor trabajo disipado por fricción, $W_{fric} = -\mu_k N L$).

#### Simulación del Trabajo en un Vórtice No Conservativo
Consideremos el campo de velocidad/fuerzas bidimensional no conservativo:

$$\mathbf{F}(x, y) = \langle -y, \ x \rangle$$

Evaluaremos numéricamente la integral de línea de $A(1, 0)$ a $B(0, 1)$ por dos trayectorias distintas:
1. **Camino Recto ($C_1$):** La línea recta segmentada $\mathbf{r}_1(t) = \langle 1-t, \ t \rangle$.
2. **Camino Curvo ($C_2$):** El arco de circunferencia del primer cuadrante $\mathbf{r}_2(t) = \langle \cos(\pi t/2), \ \sin(\pi t/2) \rangle$.

In [ ]:
def integral_linea_vectorial_discreta(F, r, t0, tf, N):
    t = np.linspace(t0, tf, N + 1)
    puntos = np.array([r(ti) for ti in t])
    
    suma = 0.0
    for i in range(N):
        t_mid = 0.5 * (t[i] + t[i+1])
        p_mid = r(t_mid)
        val_F = F(*p_mid)
        dr = puntos[i+1] - puntos[i]
        suma += np.dot(val_F, dr)
    return suma

def F_vortice(x, y):
    return np.array([-y, x])

def r_recta(t):
    return np.array([1.0 - t, t])

def r_arco(t):
    return np.array([np.cos(np.pi * t / 2.0), np.sin(np.pi * t / 2.0)])

N_particiones = 500
w_recta = integral_linea_vectorial_discreta(F_vortice, r_recta, 0.0, 1.0, N_particiones)
w_arco = integral_linea_vectorial_discreta(F_vortice, r_arco, 0.0, 1.0, N_particiones)

print("ANÁLISIS DE TRABAJO EN CAMPO NO CONSERVATIVO:")
print(f"  - Trabajo a lo largo del camino recto:       {w_recta:.6f} J  (Exacto: 1.0)")
print(f"  - Trabajo a lo largo del camino circular:    {w_arco:.6f} J  (Exacto: pi/2 = {np.pi/2:.6f})")

# Graficar los caminos
t_p = np.linspace(0, 1, 100)
xg = np.linspace(-1.5, 1.5, 12)
yg = np.linspace(-1.5, 1.5, 12)
XG, YG = np.meshgrid(xg, yg)
VX = -YG
VY = XG

plt.figure(figsize=(7, 6))
plt.quiver(XG, YG, VX, VY, color='gray', alpha=0.4, label='Campo de Fuerzas')

coords_recta = np.array([r_recta(ti) for ti in t_p])
coords_arco = np.array([r_arco(ti) for ti in t_p])

plt.plot(coords_recta[:,0], coords_recta[:,1], 'r-', linewidth=2.5, label=f'Trayecto Recto ($W={w_recta:.2f}$ J)')
plt.plot(coords_arco[:,0], coords_arco[:,1], 'b-', linewidth=2.5, label=f'Trayecto Circular ($W={w_arco:.2f}$ J)')

plt.scatter([1.0, 0.0], [0.0, 1.0], color='black', s=80, zorder=5)
plt.text(1.05, -0.05, 'A(1,0)', fontsize=10, fontweight='bold')
plt.text(-0.1, 1.05, 'B(0,1)', fontsize=10, fontweight='bold')

plt.title('Comparación de Trabajo en un Campo No Conservativo')
plt.xlabel('X')
plt.ylabel('Y')
plt.xlim(-1.5, 1.5)
plt.ylim(-1.5, 1.5)
plt.legend(loc='lower left', frameon=True, facecolor='white', framealpha=0.9)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 5. Capa de Verificación Simbólica de Integrales de Línea en SymPy

Utilizaremos `SymPy` para calcular analítica y simbólicamente la integral de línea del campo escalar original 

$$f(x, y) = x^2 + y^2$$

a lo largo de la curva $\mathbf{r}(t) = \langle t, t^2 \rangle$ en $t \in [0, 1]$, corroborando de manera exacta el valor real obtenido numéricamente ($\approx 0.99639$).

In [ ]:
t_sym = sp.Symbol('t', real=True)

# Parametrización y diferencial ds
x_c = t_sym
y_c = t_sym**2
dx_dt = x_c.diff(t_sym)
dy_dt = y_c.diff(t_sym)
ds_sym = sp.sqrt(dx_dt**2 + dy_dt**2)

# Evaluar campo en la curva
f_sym = x_c**2 + y_c**2

# Integral a evaluar
integrando = f_sym * ds_sym
integral_linea = sp.integrate(integrando, (t_sym, 0, 1))

print("Integrando en SymPy (f * ds):")
sp.pprint(integrando)
print(f"\nValor exacto simbólico calculado por SymPy: {integral_linea}")
print(f"Valor numérico aproximado de SymPy:          {integral_linea.evalf():.8f}")

assert abs(integral_linea.evalf() - 0.9963919453) < 1e-6, "La verificación de SymPy no coincide con el valor verdadero de la integral."
print("\n¡Verificación analítica de la integral de línea escalar exitosa!")

### Resumen de Conceptos Clave

1. **Integral de Línea Escalar:** Suma ponderada de un campo escalar a lo largo de una curva suave utilizando el diferencial de longitud de arco $ds = \|\mathbf{r}'(t)\| dt$.
2. **Integral de Línea Vectorial:** Acumulación del producto escalar del campo con el vector tangente diferencial $d\mathbf{r} = \mathbf{r}'(t) dt$. Define físicamente el trabajo mecánico y la circulación.
3. **Independencia de Camino:** Caracteriza a los campos conservativos (donde $\nabla \times \mathbf{F} = \mathbf{0}$). Su integral de línea depende solo de los puntos extremos y se evalúa por la diferencia de su función potencial $\phi(B) - \phi(A)$.
4. **Física del Trabajo:** En sistemas conservativos (como la gravedad) la energía se conserva. En sistemas no conservativos (como la fricción) el trabajo depende de la trayectoria y se disipa como energía térmica.

### Referencias Bibliográficas

- Boyce, W. E., & DiPrima, R. C. (2012). *Elementary Differential Equations and Boundary Value Problems*. John Wiley & Sons.
- Marsden, J. E., & Tromba, A. J. (2012). *Cálculo Vectorial* (6ta ed.). Pearson.
- Apostol, T. M. (1969). *Calculus: Vol. 2* (2da ed.). John Wiley & Sons.
- Thomas, G. B. (2014). *Cálculo: Varias Variables* (13va ed.). Pearson.